# Naive Bayes on Your Own Dataset (NLTK, standalone)

This notebook trains a **Naive Bayes text classifier** with **NLTK** on a dataset you can replace with your own. It is self-contained: run it top to bottom in Google Colab and it creates a sample dataset, trains, evaluates and lets you type your own messages.

**The sample problem:** an IT helpdesk receives short tickets. The classifier decides which team should handle each one: **access** (passwords, accounts, permissions), **hardware** (laptops, printers, screens), **network** (wifi, VPN, internet) or **software** (applications and installs).

### What you will do

| Step | What happens |
|---|---|
| 1 | Create a sample dataset, save it as a CSV, and load it back (this is where your own CSV plugs in) |
| 2 | Explore it: class balance and ticket length |
| 3 | Clean the text with NLTK: tokenise, lowercase, stopwords (keeping negations), lemmas |
| 4 | Split into training and test sets, keeping the classes balanced |
| 5 | Turn tickets into NLTK feature dictionaries |
| 6 | Train `NaiveBayesClassifier` and compare it with a "always guess the biggest class" baseline |
| 7 | Evaluate: accuracy, confusion matrix, precision, recall |
| 8 | Cross-validation, because a test set of 40 tickets is too small to trust on its own |
| 9 | Experiments: vocabulary size, bigrams, smoothing |
| 10 | Predict new tickets with a confidence score, and route uncertain ones to a person |
| 11 | Save and reload the model |
| 12 | Type your own tickets |

### How Naive Bayes works, in plain words

For each category the classifier learns two things from the training tickets: how common the category is, and how likely each word is to appear in tickets of that category. For a new ticket it multiplies those probabilities together for every word and picks the category with the biggest result.

- It is **naive** because it treats every word as independent of the others. That is not true of language, but it works well in practice
- It is fast, needs little data, and you can **see why** it decided something (the most informative words)
- A word never seen in a category would give a probability of zero and wipe out the whole result, so NLTK applies **smoothing**, which adds a tiny amount to every count

**NLTK detail that matters:** NLTK's Naive Bayes needs the feature dictionary to list **every** vocabulary word as `True` or `False`. If you only list the words that are present, it never learns from absences. On short tickets the difference is small, but on long documents such as the movie reviews in the course it cost about 15 points.

## 0. Setup

Run this cell first. It installs the libraries (Colab already has most of them) and downloads the NLTK data. If it asks you to restart the runtime, restart and run it again.

Only NLTK processes the language. pandas holds the table and matplotlib draws one chart.

In [ ]:
# 0. SETUP
%pip install -q nltk pandas matplotlib

import re
import random
import pickle
import textwrap
import warnings
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)

from nltk import NaiveBayesClassifier, FreqDist, ngrams, word_tokenize, pos_tag
from nltk.classify import accuracy
from nltk.corpus import stopwords, wordnet
from nltk.metrics import precision, recall, f_measure
from nltk.probability import LaplaceProbDist
from nltk.stem import WordNetLemmatizer

warnings.filterwarnings("ignore")
random.seed(42)                      # makes every shuffle repeatable
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 140)

print("Ready. NLTK version:", nltk.__version__)

## 1. Create the dataset (and where your own data plugs in)

A text classification dataset is a table with two columns:

| Column | Meaning |
|---|---|
| `text` | The message, ticket, review or email |
| `label` | The category a person decided it belongs to |

The cell below creates a sample dataset of **160 helpdesk tickets** (40 per category), saves it as `sample_tickets.csv`, and loads it back with pandas. Loading from a file is deliberate: it is exactly what you will do with your own data.

### Using your own dataset

1. Make a CSV with columns named `text` and `label`
2. In **Colab**, run `from google.colab import files; files.upload()` in a cell and pick your file. In **VS Code**, put the file next to the notebook
3. Set `OWN_FILE = "your_file.csv"` in the next cell and run everything again

Aim for at least **20 examples per category**, and roughly the same number in each. With fewer than about 10 per category the results will jump around.

In [ ]:
# 1. CREATE THE SAMPLE DATASET, SAVE IT, AND LOAD IT BACK
OWN_FILE = None          # set to "your_file.csv" to use your own data (needs columns: text, label)

sample = {
    "access": [
        "I forgot my password and the reset email never arrives",
        "My account is locked after too many login attempts",
        "Cannot sign in to my work email, it says the password is incorrect",
        "I need access to the finance shared folder for my new role",
        "Two factor code is not being sent to my phone",
        "Please reset my password, I am locked out",
        "My login works on my laptop but not on my phone",
        "New starter needs an account created for Monday",
        "I cannot get into the HR portal, permission denied",
        "Can you remove my old colleague's access to the project site",
        "My authenticator app stopped working after I changed phone",
        "Account suspended message when I try to log in",
        "Need admin rights to install the reporting tool",
        "I keep getting asked for my password every few minutes",
        "Unable to log in to the customer database, access denied",
        "My username has been changed and I cannot sign in",
        "Please unlock my account, I entered the wrong password three times",
        "I do not have permission to open the budget spreadsheet",
        "Single sign on page just loops back to the login screen",
        "Request access to the marketing drive for a contractor",
        "Locked out of my account since this morning",
        "How do I change my password, it expires tomorrow",
        "My password reset link has expired",
        "Need permission to edit the shared calendar",
        "Cannot log in to the portal, wrong username or password",
        "Please add me to the sales team group",
        "I am locked out of Teams after changing my password",
        "Login page says my account is disabled",
        "Need a new account for a temporary worker",
        "Access denied when I open the project folder",
        "I did not receive the verification code to sign in",
        "Please give my manager access to my mailbox while I am away",
        "My account was deactivated when I changed department",
        "Forgot my username, can you send a reminder",
        "Password reset keeps failing with an error",
        "Cannot sign in on my new phone, the security code is rejected",
        "Request read access to the legal documents site",
        "Account locked again, please unlock it",
        "I need my permissions copied from a colleague",
        "Sign in fails with an invalid credentials message",
    ],
    "hardware": [
        "My laptop screen is flickering and goes black",
        "The keyboard on my laptop has stopped working",
        "Printer on the second floor is jammed again",
        "Laptop battery only lasts twenty minutes",
        "My monitor will not turn on, the power light is off",
        "The docking station does not recognise my second screen",
        "Mouse cursor keeps freezing and the trackpad is unresponsive",
        "Need a replacement charger for my laptop",
        "The projector in meeting room three has no picture",
        "My headset microphone is not picked up by the computer",
        "Laptop is very hot and the fan is making a loud noise",
        "Printer says out of toner but I just changed it",
        "Webcam shows a black image on video calls",
        "USB ports on my desktop do not work",
        "My phone handset has a cracked screen and needs replacing",
        "The scanner in the post room is making a grinding noise",
        "Desktop computer will not start, just a beeping sound",
        "Laptop hinge is broken and the screen is loose",
        "Request a new monitor, mine has dead pixels",
        "External hard drive is not detected when I plug it in",
        "Laptop will not turn on even when plugged in",
        "The screen on my desktop is cracked",
        "My mouse stopped working, the batteries are new",
        "Printer is not printing and shows a paper jam",
        "Laptop keyboard has sticky keys after a spill",
        "Need a new headset for video calls",
        "Second monitor shows no signal",
        "My laptop is overheating and shuts down",
        "The meeting room speaker has no sound",
        "Replace the broken power cable on my docking station",
        "Camera on my laptop is not working",
        "Hard drive is making clicking noises",
        "The printer is out of paper and the tray is stuck",
        "Laptop fan runs loudly all day",
        "Phone charger port is loose and will not charge",
        "My laptop trackpad does not click any more",
        "Desk phone has no dial tone",
        "Need a laptop bag and a new mouse",
        "Screen has a black line across it",
        "The label printer prints blank labels",
    ],
    "network": [
        "Wifi keeps dropping in the east office",
        "I cannot connect to the VPN from home",
        "Internet is very slow this morning for everyone on our floor",
        "Cannot reach the shared drive, network path not found",
        "Video calls are lagging and cutting out",
        "The guest wifi network is not showing up",
        "No internet connection on my desktop but the cable is plugged in",
        "VPN connects but I cannot open any internal sites",
        "Network is down in the warehouse",
        "Website is blocked by the firewall but I need it for work",
        "Download speeds are extremely slow on the office network",
        "My IP address conflicts with another device",
        "Cannot connect to the meeting room display over wifi",
        "The remote desktop connection keeps timing out",
        "Ethernet socket by my desk is dead",
        "DNS error when opening our intranet",
        "Email works but all web pages fail to load",
        "Wifi signal is weak in the meeting rooms",
        "Unable to access cloud storage because of a network timeout",
        "Internal website unreachable from the branch office",
        "Wifi is down on the third floor",
        "Cannot connect to the office wifi with my laptop",
        "VPN keeps disconnecting every ten minutes",
        "Internet connection is extremely slow today",
        "The network drive is not accessible from my laptop",
        "Wifi asks for a password every time I connect",
        "Remote access to the office network fails",
        "No signal on the wifi in the car park",
        "Our branch has lost its internet connection",
        "Pages take forever to load on the office network",
        "Cannot connect to the printer over the network",
        "The VPN client says connection timed out",
        "Network cable is unplugged in the server room",
        "Wifi connected but no internet",
        "Video meetings freeze because the connection is unstable",
        "Router in the small office keeps restarting",
        "Cannot reach the intranet from home",
        "Slow file transfers to the shared server",
        "Firewall is blocking the supplier website",
        "Network keeps dropping when I use the VPN",
    ],
    "software": [
        "Excel crashes every time I open the monthly report",
        "Please install the latest version of Adobe Acrobat",
        "Word freezes when I try to save a large document",
        "The accounting software shows an error when I post an invoice",
        "Outlook will not open and says the profile cannot be loaded",
        "I need a licence for the design software",
        "Teams keeps closing by itself when I join a meeting",
        "My laptop has not installed the latest Windows update",
        "The CRM system is very slow when searching for customers",
        "PowerPoint file is corrupted and will not open",
        "Antivirus popup keeps appearing and blocking my application",
        "Cannot print from the browser, the button does nothing",
        "Request to install Python on my machine for data analysis",
        "The mobile app crashes on startup after the update",
        "Spreadsheet macro stopped working after the upgrade",
        "Chrome says my browser is out of date and some pages break",
        "The expenses system will not let me submit a claim",
        "Software licence has expired and the application is locked",
        "Calendar invites are not syncing between Outlook and my phone",
        "Zoom shows an error code when I start a meeting",
        "Excel will not open my file and shows an error",
        "Please install Visual Studio Code on my laptop",
        "Outlook keeps asking for my password and then crashes",
        "The application freezes when I click save",
        "Need the latest update for the payroll software",
        "Word document will not print properly",
        "Teams meeting will not start and shows an error message",
        "I need a licence for Photoshop",
        "The reporting tool crashes every time I run a report",
        "Windows update failed and my laptop restarted",
        "Browser extension is not working after the update",
        "Cannot install the new version of Zoom",
        "My PDF editor says the trial has expired",
        "Software installer fails with error code 1603",
        "Excel formulas show hash errors after the upgrade",
        "The app closes immediately after opening",
        "Need Microsoft Project installed for a new project",
        "Slack notifications stopped working after the update",
        "Database client crashes when I run a large query",
        "The finance system gives an error when I open the invoice module",
    ],
}
rows = [(text, label) for label, texts in sample.items() for text in texts]
pd.DataFrame(rows, columns=["text", "label"]).to_csv("sample_tickets.csv", index=False)

# Load from a file: the same code works for your own CSV
data = pd.read_csv(OWN_FILE or "sample_tickets.csv")
assert {"text", "label"} <= set(data.columns), "The CSV needs columns named text and label"
data = data.dropna(subset=["text", "label"]).drop_duplicates(subset="text").reset_index(drop=True)
data["text"] = data["text"].astype(str)
data["label"] = data["label"].astype(str)

print("Loaded", len(data), "rows from", OWN_FILE or "sample_tickets.csv")
data.sample(6, random_state=1)

## 2. Explore the data

Always look before you model. Three checks:

1. **Class balance.** If 90% of tickets were one category, a model that always guessed it would score 90% and learn nothing
2. **Length.** Very short tickets give the model little to work with
3. **Real examples.** Read some. Many problems are obvious to the eye (for example, a ticket that clearly belongs in two categories)

In [ ]:
# 2. EXPLORE THE DATA
counts = data["label"].value_counts()
print("Tickets per category:")
print(counts.to_string())
print(f"\nA model that always guessed '{counts.index[0]}' would score {counts.iloc[0] / len(data):.0%}. That is the number to beat.")

data["n_words"] = data["text"].apply(lambda t: len(word_tokenize(t)))
print("\nWords per ticket:", data["n_words"].describe().round(1).to_dict())

counts.sort_values().plot.barh(figsize=(5.5, 2.6), color="#4c78a8")
plt.title("Tickets per category")
plt.xlabel("Number of tickets")
plt.show()

## 3. Clean the text

We turn each ticket into a short list of useful words using the NLTK steps from the course:

1. `word_tokenize` splits the text into tokens
2. Lowercase, and keep only words made of letters
3. Remove **stopwords** (*the*, *is*, *and*), but **keep negations** such as *not* and *no*. A ticket that says "will not turn on" means something different from one that says "turns on"
4. `pos_tag` and the WordNet lemmatiser turn each word into its base form, so *crashes*, *crashing* and *crashed* all become *crash*

Check the output against a few tickets. If a cleaning step removes something important for your data, change it.

In [ ]:
# 3. CLEAN THE TEXT
lemmatizer = WordNetLemmatizer()
NEGATIONS = {"no", "nor", "not", "never", "cannot", "cant", "wont"}
STOP = set(stopwords.words("english")) - NEGATIONS

def wn_pos(tag):
    """Convert a Penn Treebank tag (from pos_tag) to the tag WordNet expects."""
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "R": wordnet.ADV}.get(tag[0], wordnet.NOUN)

def clean(text):
    """Raw text in, list of lowercase lemmas out."""
    tokens = pos_tag(word_tokenize(text))
    out = []
    for word, tag in tokens:
        w = word.lower()
        if w == "n't":
            w = "not"                              # word_tokenize splits "don't" into "do" and "n't"
        if w.isalpha() and w not in STOP:
            out.append(lemmatizer.lemmatize(w, wn_pos(tag)))
    return out

data["tokens"] = data["text"].apply(clean)

for i in [0, 45, 90, 130]:
    print(data.loc[i, "label"].upper(), "|", data.loc[i, "text"])
    print("      ->", data.loc[i, "tokens"])

## 4. Split into training and test sets

We hold some tickets back as a **test set** that the model never sees while learning. Its score tells us how the model will do on new tickets.

We split **within each category** (a *stratified* split), so both sets contain the same mix of categories. With only 160 tickets, an ordinary random split could leave a category almost missing from the test set.

The split is 75% training and 25% testing.

In [ ]:
# 4. STRATIFIED TRAIN / TEST SPLIT
def stratified_split(frame, test_share=0.25, seed=42):
    rng = random.Random(seed)
    train_idx, test_idx = [], []
    for label, group in frame.groupby("label"):
        idx = list(group.index)
        rng.shuffle(idx)
        n_test = max(1, round(len(idx) * test_share))
        test_idx += idx[:n_test]
        train_idx += idx[n_test:]
    return frame.loc[train_idx], frame.loc[test_idx]

train_df, test_df = stratified_split(data)
print("Training tickets:", len(train_df), "| Test tickets:", len(test_df))
print("\nTest set per category:", test_df["label"].value_counts().to_dict())

## 5. Turn tickets into features

NLTK classifiers learn from **feature dictionaries**, one per ticket:

```python
({"password": True, "reset": True, "wifi": False, ...}, "access")
```

Two decisions:

1. **Which words are features?** We use words that appear at least `MIN_COUNT` times in the **training** tickets. With long documents you would raise this to drop rare words. Tickets are short, so a word seen once (`projector`, `toner`) can be exactly what identifies a category, and we keep every word. Section 9 tests higher values. The vocabulary comes from the training set only: if test tickets influence it, the test score looks better than it really is (this is called **data leakage**)
2. **True and False for every word.** As explained at the start, NLTK's Naive Bayes needs the `False` entries too

In [ ]:
# 5. FEATURES
MIN_COUNT = 1            # a word must appear at least this many times in the TRAINING tickets

def build_vocab(token_lists, min_count=MIN_COUNT):
    freq = FreqDist(w for toks in token_lists for w in toks)
    return sorted(w for w, c in freq.items() if c >= min_count)

def features(tokens, vocab):
    present = set(tokens)
    return {w: (w in present) for w in vocab}

vocab = build_vocab(train_df["tokens"])
print("Vocabulary size:", len(vocab))
print("A few vocabulary words:", vocab[:15])

train_set = [(features(t, vocab), lab) for t, lab in zip(train_df["tokens"], train_df["label"])]
test_set = [(features(t, vocab), lab) for t, lab in zip(test_df["tokens"], test_df["label"])]

# What one ticket looks like to the classifier (only the True entries, for readability)
example_feats, example_label = train_set[0]
print("\nExample label:", example_label)
print("Words marked True:", [w for w, present in example_feats.items() if present])

## 6. Train the classifier

`NaiveBayesClassifier.train(train_set)` does all the learning in one line. Then we score it on the test set.

We also compute a **baseline**: the score of always guessing the most common category in the training set. A model is only useful if it clearly beats the baseline.

`show_most_informative_features` lists the words that best separate the categories. Each line reads *"when this word is present, category A is X times more likely than category B"*. It is the best way to see **why** the model decides what it decides, and to spot rubbish (a model that relies on a ticket number or a name is learning something accidental).

In [ ]:
# 6. TRAIN AND SCORE
nb = NaiveBayesClassifier.train(train_set)

test_accuracy = accuracy(nb, test_set)
majority = train_df["label"].value_counts().idxmax()
baseline = (test_df["label"] == majority).mean()

print(f"Test accuracy : {test_accuracy:.0%}  ({round(test_accuracy * len(test_set))} of {len(test_set)} tickets right)")
print(f"Baseline      : {baseline:.0%}  (always guessing '{majority}')")
print("\nThe words the model relies on most:")
nb.show_most_informative_features(12)

## 7. Evaluate properly

Accuracy is one number. The **confusion matrix** shows where the mistakes are. Each **row** is what a ticket really was, each **column** is what the model predicted. The diagonal is correct. Anything off the diagonal is a mistake, and its position tells you which two categories get mixed up.

For one category, such as `network`:

- **Precision** = of the tickets the model sent to network, how many really were network? (Low precision means the network team gets other teams' work.)
- **Recall** = of the tickets that really were network, how many did the model find? (Low recall means network problems are being sent elsewhere.)
- **F-measure** combines the two into one score

Then read the individual mistakes. They are the fastest route to improvement.

In [ ]:
# 7. CONFUSION MATRIX, PRECISION AND RECALL, AND THE MISTAKES
gold = list(test_df["label"])
pred = [nb.classify(f) for f, _ in test_set]

print(nltk.ConfusionMatrix(gold, pred).pretty_format(sort_by_count=True))

labels = sorted(set(gold))
rows = []
for lab in labels:
    ref = {i for i, g in enumerate(gold) if g == lab}
    tst = {i for i, p in enumerate(pred) if p == lab}
    rows.append((lab, len(ref),
                 round(precision(ref, tst) or 0, 2), round(recall(ref, tst) or 0, 2), round(f_measure(ref, tst) or 0, 2)))
print(pd.DataFrame(rows, columns=["category", "tickets", "precision", "recall", "F"]).to_string(index=False))

# The tickets the model got wrong, with how sure it was
print("\nMistakes:")
mistakes = 0
for (feats, true), text in zip(test_set, test_df["text"]):
    dist = nb.prob_classify(feats)
    if dist.max() != true:
        mistakes += 1
        print(f"  actual {true:<9} predicted {dist.max():<9} ({dist.prob(dist.max()):.0%} sure)  {text}")
if mistakes == 0:
    print("  None on this split. With only 40 test tickets, do not read too much into that. See section 8.")

## 8. Cross-validation: a more trustworthy score

With 40 test tickets, one ticket is worth 2.5 percentage points. A lucky or unlucky split could move the score a lot.

**Cross-validation** fixes this. We split the data into 5 parts (**folds**). We train on four, test on the fifth, and rotate until every part has been the test set once. The average of the five scores is a steadier estimate, and the spread tells you how much to trust it.

Each fold rebuilds the vocabulary from its own training part, so no test information leaks in.

In [ ]:
# 8. 5-FOLD CROSS-VALIDATION (vocabulary rebuilt inside every fold)
def make_folds(frame, k=5, seed=42):
    rng = random.Random(seed)
    folds = [[] for _ in range(k)]
    for label, group in frame.groupby("label"):
        idx = list(group.index)
        rng.shuffle(idx)
        for n, i in enumerate(idx):
            folds[n % k].append(i)
    return folds

def cross_validate(frame, k=5, min_count=MIN_COUNT, use_bigrams=False, estimator=None, seed=42):
    folds = make_folds(frame, k, seed)
    scores = []
    for n in range(k):
        test_i = folds[n]
        train_i = [i for m in range(k) if m != n for i in folds[m]]
        tok = (lambda toks: toks + [a + "_" + b for a, b in ngrams(toks, 2)]) if use_bigrams else (lambda toks: toks)
        tr_tokens = [tok(frame.loc[i, "tokens"]) for i in train_i]
        v = build_vocab(tr_tokens, min_count)
        tr = [(features(t, v), frame.loc[i, "label"]) for t, i in zip(tr_tokens, train_i)]
        te = [(features(tok(frame.loc[i, "tokens"]), v), frame.loc[i, "label"]) for i in test_i]
        model = NaiveBayesClassifier.train(tr, estimator=estimator) if estimator else NaiveBayesClassifier.train(tr)
        scores.append(accuracy(model, te))
    return scores

scores = cross_validate(data)
mean = sum(scores) / len(scores)
spread = (sum((s - mean) ** 2 for s in scores) / len(scores)) ** 0.5
print("Accuracy per fold:", [f"{s:.0%}" for s in scores])
print(f"Average {mean:.0%}, typical spread of about {spread:.0%} either side")

## 9. Experiments: what improves the model?

Change **one thing at a time** and compare with cross-validation. With a small dataset, treat differences of a couple of points as noise.

| Experiment | Idea |
|---|---|
| **Minimum count** | Dropping rare words removes noise in long documents, but removes real signal when tickets are short. Expect accuracy to fall as you raise it |
| **Bigrams** | Adds word pairs such as `not work` or `wifi drop`, so a little word order is kept |
| **Smoothing** | The default (expected likelihood) against Laplace smoothing. Often almost no difference |

The most important lever is usually **more and better data**, not a cleverer setting.

In [ ]:
# 9. EXPERIMENTS (all scored by 5-fold cross-validation)
experiments = [
    ("Default (min count 1, unigrams)",    dict()),
    ("Drop rare words (min count 2)",      dict(min_count=2)),
    ("Drop rare words (min count 3)",      dict(min_count=3)),
    ("Unigrams + bigrams",                 dict(use_bigrams=True)),
    ("Laplace smoothing",                  dict(estimator=LaplaceProbDist)),
]
rows = []
for name, kwargs in experiments:
    s = cross_validate(data, **kwargs)
    rows.append((name, round(sum(s) / len(s), 3)))
results = pd.DataFrame(rows, columns=["setting", "cv accuracy"])
print(results.to_string(index=False))

results.set_index("setting")["cv accuracy"].sort_values().plot.barh(figsize=(7, 2.8), color="#2a9d8f")
plt.xlim(0, 1)
plt.title("Cross-validated accuracy by setting")
plt.xlabel("Accuracy")
plt.tight_layout()
plt.show()

## 10. Predict new tickets, with a confidence score

`prob_classify` returns a probability for every category, not just the winner. We use it two ways:

1. **Show the confidence**, so a person can see how sure the model was
2. **Set an "unsure" band.** If the top probability is below a threshold, the ticket goes to a person instead of being routed automatically. Naive Bayes tends to be overconfident (it often says 99% when it is wrong), so keep the threshold high and never treat the probability as exact

The last two tickets below are deliberately ambiguous. Look at how the model handles them.

In [ ]:
# 10. PREDICT NEW TICKETS
UNSURE_BELOW = 0.70          # below this top probability, a person decides

def predict(text):
    """Return (category or 'unsure', top probability, all probabilities)."""
    feats = features(clean(text), vocab)
    dist = nb.prob_classify(feats)
    best = dist.max()
    probs = {lab: round(dist.prob(lab), 2) for lab in sorted(dist.samples())}
    return (best if dist.prob(best) >= UNSURE_BELOW else "unsure"), dist.prob(best), probs

new_tickets = [
    "My laptop will not charge and the battery light is off",
    "I cannot log in, it says my password has expired",
    "The wifi is very slow in the canteen",
    "Excel keeps crashing when I open large files",
    "Need a licence for a new PDF tool",
    "VPN rejects my password",                          # network or access?
    "The printer app will not install on my laptop",    # hardware or software?
]
for text in new_tickets:
    category, top, probs = predict(text)
    print(f"{category:<9} ({top:.0%})  {text}")
    print("          ", probs)

## 11. Save and reload the model

`pickle` writes the trained classifier and its vocabulary to a file, so you can reuse them without retraining. Always save the **vocabulary with the model**: the model is useless without the exact word list it was trained on.

In [ ]:
# 11. SAVE AND RELOAD
with open("ticket_classifier.pkl", "wb") as f:
    pickle.dump({"model": nb, "vocab": vocab, "unsure_below": UNSURE_BELOW}, f)

with open("ticket_classifier.pkl", "rb") as f:
    saved = pickle.load(f)

def predict_saved(text):
    feats = features(clean(text), saved["vocab"])
    dist = saved["model"].prob_classify(feats)
    best = dist.max()
    return (best if dist.prob(best) >= saved["unsure_below"] else "unsure"), round(dist.prob(best), 2)

print(predict_saved("The projector in the boardroom has no picture"))
print(predict_saved("Please give my new starter access to the shared folder"))

## 12. Type your own tickets

Run the cell and type a message into the box. The classifier says which category it chooses, how sure it is, and which of your words it recognised. Type `quit` to stop.

Set `LIVE_INPUT = False` if you want "Run all" to finish without waiting for you.

In [ ]:
# 12. LIVE INPUT
LIVE_INPUT = True

def live():
    print("Type a ticket and press Enter. Type 'quit' to stop.")
    while True:
        try:
            text = input("\nTicket: ").strip()
        except (EOFError, KeyboardInterrupt):
            break
        if text.lower() in {"quit", "exit", "bye"}:
            print("Goodbye!")
            break
        if not text:
            continue
        category, top, probs = predict(text)
        known = [w for w in clean(text) if w in set(vocab)]
        print(f"  -> {category.upper()}  ({top:.0%} sure)")
        print("  probabilities:", probs)
        print("  words the model recognised:", known if known else "none (so it is guessing from category frequency alone)")

if LIVE_INPUT:
    live()

## Where to go next

**Use your own data.** Set `OWN_FILE` in section 1 and run everything again. Check the class balance first, then read the most informative features: if you see names, dates or ticket numbers there, remove them in the `clean` function.

**Improve the model.**

| Problem you see | What to try |
|---|---|
| One category scores badly | Add more examples of it. Read its mistakes in section 7 |
| Two categories keep getting mixed up | Check whether your labels are consistent. Two people may label the same ticket differently |
| Accuracy is high but real tickets go wrong | Your sample does not look like real tickets. Collect real ones |
| The model is overconfident | Use a higher `UNSURE_BELOW`, or compare with a model that gives better probabilities, such as logistic regression |

**Limits of Naive Bayes.** It treats words as independent, so it cannot tell *"printer app will not install"* (software) from *"printer will not turn on"* (hardware) unless the words themselves carry the difference. It cannot read meaning, only words. Logistic regression or a support vector machine usually score a few points higher, and NLTK can run both through `nltk.classify.SklearnClassifier`.

### Exercises

1. **Add a fifth category**, such as `facilities`, with 20 tickets of your own. Rerun everything. Does the confusion matrix show new mix-ups?
2. **Break it on purpose.** Build the vocabulary from all tickets (train and test together) and compare the test accuracy. By how much does leakage flatter the model?
3. **Count the effect of stopwords.** In `clean`, stop removing stopwords. Does cross-validated accuracy change?
4. **Tune the unsure band.** Try `UNSURE_BELOW` at 0.5, 0.7 and 0.9 on the test set. How many tickets go to a person at each level, and how accurate are the rest?
5. **Try a different label.** Use the same tickets to predict urgency (high, normal, low) by relabelling them. Is it easier or harder than predicting the team?